# Retrieval
## Loading model

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2196.83it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Database connection

In [2]:
from pathlib import Path
import chromadb

PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"

# Retrieval of recursive chunking

In [4]:
chroma_client = chromadb.PersistentClient(path=str(DATA_DIR / "chromadb"))
colRecursive = chroma_client.get_collection(name="recursive_100")

print(f"Collection has {colRecursive.count()} chunks")

Collection has 31741 chunks


In [5]:
query = "What is Zostera marina?"
query_embedding = embedding_model.embed_query(query)

results = colRecursive.query(
    query_embeddings=[query_embedding],
    n_results=5,
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"--- {meta['filename']} (distance: {dist:.4f}) ---")
    print(f"Title: {meta['title']}")
    print(f"Authors: {meta['authors']}")
    print(doc[:300])
    print()

--- algae-2009-24-3-179.pdf (distance: 0.5951) ---
Title: Deep Learning for Image Recognition
Authors: ...
Zostera marina L. is an important species in coastal ecosystems because it contributes for nutrient cycling and sediment stabilizer, and provides food stuffs and habitat for many marine organisms such as invertebrates

--- algae-2002-17-2-071.pdf (distance: 0.6157) ---
Title: Zostera geojeensis, a New Species of Seagrass from Korea
Authors: Hyunchur Shin1, Kang-Hyun Cho2, Yoon Sik Oh3, 1Division of Biological Sciences, Soonchunhyang University, Asan, 3Division of Life Sciences, Gyeongsang National University, Chinju
The genus Zostera L. (Zosteraceae), one of the seagrass genera, consists of 11 species, and are widely distributed in the northern and the southern temperate waters (Hartog 1970). The genus Zostera is divided into two sub- genera, Zostera and Zosterella, based on the closed or open sheath in leaf ba

--- algae-2013-28-3-267.pdf (distance: 0.6306) ---
Title: Marine macr

In [10]:
import json

recursive_results = {
    "query": "What is Zostera marina?",
    "strategy": "recursive_1000",
    "results": [
        {"filename": meta["filename"], "title": meta["title"], "authors": meta["authors"], "distance": dist, "text": doc}
        for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]
}

with open(DATA_DIR / "test" / "retrieval_recursive.json", "w", encoding="utf-8") as f:
    json.dump(recursive_results, f, ensure_ascii=False, indent=2)

## Retrieval of Semantic

On the test query, the retrieved chunks were not as good, the distances are 0.71 vs around 0.59-0.63

In [6]:
chroma_client = chromadb.PersistentClient(path=str(DATA_DIR / "chromadb"))
colSemantic= chroma_client.get_collection(name="semantic_p95")

print(f"Collection has {colSemantic.count()} chunks")

Collection has 14741 chunks


In [7]:
query = "What is Zostera marina?"
query_embedding = embedding_model.embed_query(query)

results = colSemantic.query(
    query_embeddings=[query_embedding],
    n_results=5,
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"--- {meta['filename']} (distance: {dist:.4f}) ---")
    print(f"Title: {meta['title']}")
    print(f"Authors: {meta['authors']}")
    print(doc[:300])
    print()

--- algae-2005-20-4-379.pdf (distance: 0.6888) ---
Title: Selection of the Optimal Transplanting Method and Time for Restoration of Zostera marina Habitats
Authors: Jung-Im Park, Young-Kyun Kim, Sang Rul Park, Jong-Hyeob Kim, Young-Sang Kim, Jeong-Bae Kim, Pil-Yong Lee, Chang-Keun Kang, Kun-Seop Lee
Algae Volume 20(4): 379-388, 2005

잘피(Zostera marina)서식지 복원을 위한 최적 이식방법 및 시기 선정에 관한 연구

박정임1∙김영균1∙박상률1∙김종협1∙김영상1∙김정배1,2∙이필용2∙강창근1∙이근섭1*

(1부산대학교 생물학과∙2국립수산과학원 남해수산연구소)

Selection of the Optimal Transplanting Method and Time for Restoration of Zostera marina Habitats

Jung-Im Park1, Young-Kyun Kim1, 

--- algae-2009-24-3-179.pdf (distance: 0.7062) ---
Title: Deep Learning for Image Recognition
Authors: ...
2002). Therefore, the identification and isolation of new natural antioxidants from aquatic and terrestrial plants are desir- able (Nishida et al. 1996). Zostera marina L. is an important species in coastal ecosystems because it contributes for nutrient cycling and sediment stabilizer, and

In [11]:
semantic_results = {
    "query": "What is Zostera marina?",
    "strategy": "semantic_p95",
    "results": [
        {"filename": meta["filename"], "title": meta["title"], "authors": meta["authors"], "distance": dist, "text": doc}
        for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]
}

with open(DATA_DIR / "test" / "retrieval_semantic.json", "w", encoding="utf-8") as f:
    json.dump(semantic_results, f, ensure_ascii=False, indent=2)